# Manual Mapping 

In this notebook, I am testing a manual mapping workflow for residential solar panel mapping using Google My Maps.
The goal is to explore how manually drawn polygons and points from Google My Maps can be extracted, structured and analyzed for solar potential assessments. This includes:

- Creating and exporting custom maps from Google My Maps (e.g., KML/KMZ format)
- Parsing the geographic features (points, polygons) into a structured format (DataFrame/GeoDataFrame)
- Evaluating the feasibility of using this method as a simple, low-cost alternative to automated solar mapping tools.

## Setup 

In [111]:
import pandas as pd
import geopandas as gpd
import xml.etree.ElementTree as ET
import zipfile
from shapely.geometry import Polygon
import os
from datetime import datetime

## Data processing 

### KML files to Dataframe

In [112]:
def kmz_to_dataframe(kmz_file):
    # extract KML file inside KMZ (ZIP archive)
    with zipfile.ZipFile(kmz_file, 'r') as kmz:
        kml_filename = [f for f in kmz.namelist() if f.endswith('.kml')][0]
        with kmz.open(kml_filename) as kml_file:
            tree = ET.parse(kml_file)

    root = tree.getroot()
    ns = {"kml": "http://www.opengis.net/kml/2.2"}

    data = []
    for placemark in root.findall(".//kml:Placemark", ns):
        name = placemark.find("kml:name", ns)
        description = placemark.find("kml:description", ns)
        polygon = placemark.find(".//kml:Polygon", ns)

        if polygon is not None:
            coords = polygon.find(".//kml:outerBoundaryIs/kml:LinearRing/kml:coordinates", ns)
            if coords is not None:
                # Parse each coordinate pair
                coord_list = []
                for coord in coords.text.strip().split():
                    lon, lat, *alt = coord.split(",")
                    coord_list.append((float(lon), float(lat)))

                data.append({
                    "ID": name.text if name is not None else None,
                    "description": description.text if description is not None else None,
                    "polygon": coord_list
                })

    return pd.DataFrame(data)


In [113]:
df_1= kmz_to_dataframe("../../Google My Maps/Solar_Panel _China_1.kmz")
df_1

,ID,description,polygon
0,东夹河村委会1,None,"[(115.4524716, 36.2657547), (115.4524662, 36.2..."
1,东夹河村委会2,None,"[(115.4522007, 36.265519), (115.452198, 36.265..."
2,东夹河村委会3,None,"[(115.4523509, 36.2651102), (115.4523509, 36.2..."
3,东夹河村委会4,None,"[(115.4525816, 36.2643944), (115.4525735, 36.2..."
4,东夹河村委会5,None,"[(115.4527718, 36.2631591), (115.452753, 36.26..."
...,...,...,...
195,苗未城村委会11,None,"[(115.1048438, 36.2541323), (115.1048438, 36.2..."
196,苗未城村委会 12,None,"[(115.1040123, 36.2537197), (115.1040069, 36.2..."
197,苗未城村委会13,None,"[(115.1035135, 36.2535844), (115.1035215, 36.2..."
198,苗未城村委会14,Cropland,"[(115.1018532, 36.2543804), (115.1017942, 36.2..."


In [114]:
def fill_description(df, col="description", fill_value="Residential"):
    """
    Replace empty or missing values in the description column with a default value.
    
    Parameters:
        df (pd.DataFrame): Input DataFrame.
        col (str): Column name to clean (default 'description').
        fill_value (str): Value to use when description is empty/missing.
    
    Returns:
        pd.DataFrame: DataFrame with cleaned description column.
    """
    df = df.copy()
    df[col] = df[col].fillna(fill_value)        # replace NaN
    df[col] = df[col].replace("", fill_value)   # replace empty strings
    return df


In [115]:
df_1 = fill_description(df_1)

In [116]:
print(df_1.head())
df_1.info()

        ID  description                                            polygon
0  东夹河村委会1  Residential  [(115.4524716, 36.2657547), (115.4524662, 36.2...
1  东夹河村委会2  Residential  [(115.4522007, 36.265519), (115.452198, 36.265...
2  东夹河村委会3  Residential  [(115.4523509, 36.2651102), (115.4523509, 36.2...
3  东夹河村委会4  Residential  [(115.4525816, 36.2643944), (115.4525735, 36.2...
4  东夹河村委会5  Residential  [(115.4527718, 36.2631591), (115.452753, 36.26...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           200 non-null    object
 1   description  200 non-null    object
 2   polygon      200 non-null    object
dtypes: object(3)
memory usage: 4.8+ KB


In [117]:
def polygons_to_geodataframe(df, polygon_col="polygon", crs="EPSG:4326", project_crs="EPSG:3857"):
    """
    Convert a DataFrame with polygon coordinates into a GeoDataFrame,
    compute centroids safely and calculate area.
    
    Parameters:
        df (pd.DataFrame): Input DataFrame with a column of polygons (list of (lon, lat) tuples).
        polygon_col (str): Column containing polygon coordinate lists.
        crs (str): CRS of input coordinates (default WGS84 EPSG:4326).
        project_crs (str): Projected CRS for centroid and area calculation (default Web Mercator EPSG:3857).
    
    Returns:
        gpd.GeoDataFrame: GeoDataFrame with geometry, centroid lat/lon, and area in square meters.
    """
    df = df.copy()
    df["geometry"] = df[polygon_col].apply(lambda coords: Polygon(coords))
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs=crs)
    
    # Reproject to projected CRS for centroid and area calculation
    gdf_proj = gdf.to_crs(project_crs)
    
    # Centroid
    centroids = gdf_proj.centroid.to_crs(crs)
    gdf["lat"] = centroids.y
    gdf["lon"] = centroids.x
    
    
    # Area in projected CRS (square meters)
    gdf["area_m2"] = gdf_proj.geometry.area
    
    return gdf


In [118]:
gdf_1= polygons_to_geodataframe(df_1)
gdf_1

,ID,description,polygon,geometry,lat,lon,area_m2
0,东夹河村委会1,Residential,"[(115.4524716, 36.2657547), (115.4524662, 36.2...","POLYGON ((115.45 36.266, 115.45 36.266, 115.45...",36.265719,115.452585,187.122044
1,东夹河村委会2,Residential,"[(115.4522007, 36.265519), (115.452198, 36.265...","POLYGON ((115.45 36.266, 115.45 36.265, 115.45...",36.265484,115.452294,194.365490
2,东夹河村委会3,Residential,"[(115.4523509, 36.2651102), (115.4523509, 36.2...","POLYGON ((115.45 36.265, 115.45 36.265, 115.45...",36.265072,115.452459,203.712330
3,东夹河村委会4,Residential,"[(115.4525816, 36.2643944), (115.4525735, 36.2...","POLYGON ((115.45 36.264, 115.45 36.264, 115.45...",36.264356,115.452654,131.867804
4,东夹河村委会5,Residential,"[(115.4527718, 36.2631591), (115.452753, 36.26...","POLYGON ((115.45 36.263, 115.45 36.263, 115.45...",36.263127,115.452876,235.251860
...,...,...,...,...,...,...,...
195,苗未城村委会11,Residential,"[(115.1048438, 36.2541323), (115.1048438, 36.2...","POLYGON ((115.1 36.254, 115.1 36.254, 115.11 3...",36.254099,115.104930,95.074208
196,苗未城村委会 12,Residential,"[(115.1040123, 36.2537197), (115.1040069, 36.2...","POLYGON ((115.1 36.254, 115.1 36.254, 115.1 36...",36.253692,115.104095,84.977532
197,苗未城村委会13,Residential,"[(115.1035135, 36.2535844), (115.1035215, 36.2...","POLYGON ((115.1 36.254, 115.1 36.254, 115.1 36...",36.253534,115.103624,200.647734
198,苗未城村委会14,Cropland,"[(115.1018532, 36.2543804), (115.1017942, 36.2...","POLYGON ((115.1 36.254, 115.1 36.254, 115.1 36...",36.254132,115.102365,5446.471869


### Get the Village 

In [119]:
village = pd.read_excel("../../List_villages.xlsx")
village.head()

,Province,Province_pinyin,Province code,City,City_pinyin,City code,County,County_pinyin,County code,Township,Township_pinyin,Village,Village_pinyin,Unnamed: 13
0,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,北峰乡,bei feng xiang,东夹河村委会,dong jia he cun wei hui,https://www.google.fr/maps/d/u/0/edit?mid=18TO...
1,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,埝头乡,nian tou xiang,朱村村村委会,zhu cun cun cun wei hui,NaN
2,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,大名镇,da ming zhen,吴水坑村委会,wu shui keng cun wei hui,NaN
3,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,大街乡,da jie xiang,张郭村委会,zhang guo cun wei hui,NaN
4,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,张铁集乡,zhang tie ji xiang,北刘店村委会,bei liu dian cun wei hui,NaN


In [120]:
def enrich_with_village_info(gdf, village, gdf_name_col="ID", village_name_col="Village"):
    """
    Enrich a GeoDataFrame with all columns from a village database based on partial name matching.
    
    Parameters:
        gdf (GeoDataFrame): Polygons with a 'name' column.
        village (DataFrame): Village database with 'village' and other columns.
        gdf_name_col (str): Column in gdf containing polygon names (default 'ID').
        village_name_col (str): Column in village_db containing village names (default 'village').
    
    Returns:
        GeoDataFrame: gdf enriched with all columns from village_db where village_name is contained in polygon name.
    """
    gdf = gdf.copy()
    
    enriched_rows = []

    for idx, row in gdf.iterrows():
        # Find all the villages wich is contains in row[name]
        matches = village[village[village_name_col].apply(lambda x: x in row[gdf_name_col])]
        
        if not matches.empty:
            # If there is more than one mach we take the first 
            village_info = matches.iloc[0].to_dict()
        else:
            # Else fill with NaN 
            village_info = {col: pd.NA for col in village.columns}
        
        # Combine gdf and village
        combined = { **village_info,**row.to_dict()}
        enriched_rows.append(combined)
    
    enriched_gdf = pd.DataFrame(enriched_rows)
    
    # Convert to GeoDataFrame if 'geometry' existis 
    if 'geometry' in enriched_gdf.columns:
        import geopandas as gpd
        enriched_gdf = gpd.GeoDataFrame(enriched_gdf, geometry='geometry', crs=gdf.crs)
    
    return enriched_gdf


In [121]:
enriched_gdf_1 = enrich_with_village_info(gdf_1, village)
enriched_gdf_1

,Province,Province_pinyin,Province code,City,City_pinyin,City code,County,County_pinyin,County code,Township,...,Village,Village_pinyin,Unnamed: 13,ID,description,polygon,geometry,lat,lon,area_m2
0,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,北峰乡,...,东夹河村委会,dong jia he cun wei hui,https://www.google.fr/maps/d/u/0/edit?mid=18TO...,东夹河村委会1,Residential,"[(115.4524716, 36.2657547), (115.4524662, 36.2...","POLYGON ((115.45 36.266, 115.45 36.266, 115.45...",36.265719,115.452585,187.122044
1,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,北峰乡,...,东夹河村委会,dong jia he cun wei hui,https://www.google.fr/maps/d/u/0/edit?mid=18TO...,东夹河村委会2,Residential,"[(115.4522007, 36.265519), (115.452198, 36.265...","POLYGON ((115.45 36.266, 115.45 36.265, 115.45...",36.265484,115.452294,194.365490
2,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,北峰乡,...,东夹河村委会,dong jia he cun wei hui,https://www.google.fr/maps/d/u/0/edit?mid=18TO...,东夹河村委会3,Residential,"[(115.4523509, 36.2651102), (115.4523509, 36.2...","POLYGON ((115.45 36.265, 115.45 36.265, 115.45...",36.265072,115.452459,203.712330
3,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,北峰乡,...,东夹河村委会,dong jia he cun wei hui,https://www.google.fr/maps/d/u/0/edit?mid=18TO...,东夹河村委会4,Residential,"[(115.4525816, 36.2643944), (115.4525735, 36.2...","POLYGON ((115.45 36.264, 115.45 36.264, 115.45...",36.264356,115.452654,131.867804
4,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,北峰乡,...,东夹河村委会,dong jia he cun wei hui,https://www.google.fr/maps/d/u/0/edit?mid=18TO...,东夹河村委会5,Residential,"[(115.4527718, 36.2631591), (115.452753, 36.26...","POLYGON ((115.45 36.263, 115.45 36.263, 115.45...",36.263127,115.452876,235.251860
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,西未庄乡,...,苗未城村委会,miao wei cheng cun wei hui,NaN,苗未城村委会11,Residential,"[(115.1048438, 36.2541323), (115.1048438, 36.2...","POLYGON ((115.1 36.254, 115.1 36.254, 115.11 3...",36.254099,115.104930,95.074208
196,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,西未庄乡,...,苗未城村委会,miao wei cheng cun wei hui,NaN,苗未城村委会 12,Residential,"[(115.1040123, 36.2537197), (115.1040069, 36.2...","POLYGON ((115.1 36.254, 115.1 36.254, 115.1 36...",36.253692,115.104095,84.977532
197,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,西未庄乡,...,苗未城村委会,miao wei cheng cun wei hui,NaN,苗未城村委会13,Residential,"[(115.1035135, 36.2535844), (115.1035215, 36.2...","POLYGON ((115.1 36.254, 115.1 36.254, 115.1 36...",36.253534,115.103624,200.647734
198,河北,Hebei,13,邯郸市,Handan,1304,大名县,Daming xian,130425,西未庄乡,...,苗未城村委会,miao wei cheng cun wei hui,NaN,苗未城村委会14,Cropland,"[(115.1018532, 36.2543804), (115.1017942, 36.2...","POLYGON ((115.1 36.254, 115.1 36.254, 115.1 36...",36.254132,115.102365,5446.471869


## Database

Loops through all KMZ files in the folder "Google My Maps".
Converts each KMZ into a clean GeoDataFrame (using your polygons_to_geodataframe).
Enriches with your village database using the enrich_with_village_info.
Concatenates all results into one big DataFrame.
Saves the final dataset as an Excel file.

In [122]:
# Pipeline to process all KMZ files in the folder 
def process_kmz_folder(folder_path, village, output_excel):
    all_gdfs = []

    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
    name, ext = os.path.splitext(output_excel)
    output_excel = f"{name}_{timestamp}{ext}"
    
    for file in os.listdir(folder_path):
        if file.endswith(".kmz"):
            kmz_path = os.path.join(folder_path, file)
            print(f"Processing {kmz_path}...")
            
            # Extract KMZ → DataFrame (you need your kmz_polygons_to_dataframe function here) + add description
            df = kmz_to_dataframe(kmz_path)  
            df = fill_description(df)
            
            # Convert polygons to GeoDataFrame
            gdf = polygons_to_geodataframe(df)
            
            # Enrich with village info
            gdf = enrich_with_village_info(gdf, village)
            
            # Collect
            all_gdfs.append(gdf)
    
    # Merge all together
    merged_gdf = pd.concat(all_gdfs, ignore_index=True)
    
    # Save as Excel
    merged_gdf.drop(columns="geometry").to_excel(output_excel, index=False)
    print(f"Saved merged dataset to {output_excel}")
    
    return merged_gdf

In [135]:
# Path to folder of Google My Maps exports
folder_path = "../../Google My Maps"

# Run pipeline
merged_gdf = process_kmz_folder(folder_path, village, "../../solar_panel_manual_maps.xlsx")


Processing ../../Google My Maps\Solar_Panel _China_1.kmz...
Processing ../../Google My Maps\Solar_Panel_China_2.kmz...
Processing ../../Google My Maps\Solar_Panel_China_3.kmz...
Processing ../../Google My Maps\Solar_Panel_China_4.kmz...
Processing ../../Google My Maps\Solar_Panel_China_5.kmz...
Processing ../../Google My Maps\Solar_Panel_China_6.kmz...
Saved merged dataset to ../../solar_panel_manual_maps_2025-08-31_1649.xlsx
